# 04. Vector Search and Cosine Similarity — `proyecto_integrador_v2`

Este notebook corresponde al **Paso 04** del pipeline `proyecto_integrador_v2`.

## Objetivo

Construir una búsqueda visual Top-K usando los embeddings generados en el Paso 03.

En esta versión del proyecto, el objetivo principal no es encontrar perros de la misma raza, sino encontrar **perros visualmente similares**. Esto es la base para un sistema de perros perdidos/encontrados, donde una imagen nueva podrá compararse contra una base de perros reportados.

## Entrada

Este notebook usa:

```text
processed_data/embeddings/step03_dog_embeddings_l2.npy
processed_data/metadata/step03_embeddings_metadata.csv
```

## Salidas

El notebook generará:

```text
processed_data/search_results/step04_topk_neighbors.csv
processed_data/search_results/step04_similarity_summary.csv
reports/tables/step04_vector_search_indicators.csv
reports/figures/step04_similarity_examples.png
```

## Enfoque

Como los embeddings ya están normalizados con L2, la similitud coseno entre dos embeddings se puede calcular como producto punto:

```text
cosine_similarity(a, b) = a · b
```

El resultado va de menor a mayor similitud. Para este pipeline:

- similitud alta → perros visualmente más parecidos,
- similitud baja → perros visualmente menos parecidos.

Este paso construye una búsqueda Top-K para encontrar los vecinos más similares de cada imagen.

In [ ]:
# 0. Instalación de dependencias

!pip install -q numpy pandas matplotlib pillow tqdm scikit-learn

In [ ]:
# 1. Imports y configuración general

from pathlib import Path
import json
import time
import math
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 200)

SEED = 42
np.random.seed(SEED)

In [ ]:
# 2. Montar Google Drive

from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 3. Rutas del proyecto

PROJECT_ROOT = Path("/content/drive/MyDrive/proyecto_integrador_v2")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    config = {}

PROCESSED_DATA_PATH = PROJECT_ROOT / "processed_data"
EMBEDDINGS_PATH = PROCESSED_DATA_PATH / "embeddings"
METADATA_PATH = PROCESSED_DATA_PATH / "metadata"
SEARCH_RESULTS_PATH = PROCESSED_DATA_PATH / "search_results"

REPORTS_PATH = PROJECT_ROOT / "reports"
FIGURES_PATH = REPORTS_PATH / "figures"
TABLES_PATH = REPORTS_PATH / "tables"

for p in [SEARCH_RESULTS_PATH, REPORTS_PATH, FIGURES_PATH, TABLES_PATH]:
    p.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_L2_PATH = EMBEDDINGS_PATH / "step03_dog_embeddings_l2.npy"
EMBEDDINGS_METADATA_PATH = METADATA_PATH / "step03_embeddings_metadata.csv"

TOP_K = 10
VISUAL_TOP_K = 5

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EMBEDDINGS_L2_PATH:", EMBEDDINGS_L2_PATH)
print("EMBEDDINGS_METADATA_PATH:", EMBEDDINGS_METADATA_PATH)
print("TOP_K:", TOP_K)

In [ ]:
# 4. Cargar embeddings y metadata del Paso 03

if not EMBEDDINGS_L2_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {EMBEDDINGS_L2_PATH}. Ejecuta primero el Paso 03."
    )

if not EMBEDDINGS_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {EMBEDDINGS_METADATA_PATH}. Ejecuta primero el Paso 03."
    )

embeddings_l2 = np.load(EMBEDDINGS_L2_PATH)
metadata_df = pd.read_csv(EMBEDDINGS_METADATA_PATH)

metadata_df = metadata_df.reset_index(drop=True)

if len(metadata_df) != embeddings_l2.shape[0]:
    print("Advertencia: metadata y embeddings tienen tamaños diferentes.")
    print("metadata:", len(metadata_df))
    print("embeddings:", embeddings_l2.shape[0])

    min_len = min(len(metadata_df), embeddings_l2.shape[0])
    metadata_df = metadata_df.head(min_len).copy()
    embeddings_l2 = embeddings_l2[:min_len]

print("Embeddings:", embeddings_l2.shape)
print("Metadata:", metadata_df.shape)

display(metadata_df.head())

In [ ]:
# 5. Validación de embeddings normalizados

norms = np.linalg.norm(embeddings_l2, axis=1)

print("Norma mínima:", norms.min())
print("Norma máxima:", norms.max())
print("Norma promedio:", norms.mean())

if np.allclose(norms.mean(), 1.0, atol=1e-3):
    print("OK: embeddings normalizados con L2.")
else:
    print("Advertencia: revisar normalización L2.")

## Selección de muestra para búsqueda

Calcular vecinos para las 19,153 imágenes contra toda la base es posible, pero puede consumir memoria y tiempo.

Por eso, este notebook usa dos modos:

1. **Modo muestra**: calcula Top-K para una muestra aleatoria.
2. **Modo completo**: calcula Top-K para toda la base por chunks.

Por default dejamos activado el modo completo por chunks, que es más seguro en memoria.

In [ ]:
# 6. Configuración de búsqueda Top-K

RUN_FULL_SEARCH = True
QUERY_SAMPLE_SIZE = 2000
CHUNK_SIZE = 512

if RUN_FULL_SEARCH:
    query_indices = np.arange(embeddings_l2.shape[0])
else:
    sample_size = min(QUERY_SAMPLE_SIZE, embeddings_l2.shape[0])
    query_indices = np.random.choice(embeddings_l2.shape[0], size=sample_size, replace=False)

print("Queries a procesar:", len(query_indices))
print("Base de búsqueda:", embeddings_l2.shape[0])
print("CHUNK_SIZE:", CHUNK_SIZE)

In [ ]:
# 7. Funciones de búsqueda Top-K por similitud coseno

def topk_cosine_search(query_vectors, database_vectors, query_indices, top_k=10):
    """Calcula Top-K vecinos para un batch de queries.

    Como los embeddings están normalizados L2, producto punto = similitud coseno.
    """
    sims = np.dot(query_vectors, database_vectors.T)

    rows = []

    for local_i, global_query_idx in enumerate(query_indices):
        sim_row = sims[local_i]

        # Excluir self-match si el query viene de la misma base
        sim_row = sim_row.copy()
        sim_row[global_query_idx] = -np.inf

        top_indices = np.argsort(sim_row)[::-1][:top_k]

        query_meta = metadata_df.iloc[global_query_idx]

        for rank, neighbor_idx in enumerate(top_indices, start=1):
            neighbor_meta = metadata_df.iloc[int(neighbor_idx)]

            rows.append({
                "query_index": int(global_query_idx),
                "neighbor_rank": int(rank),
                "neighbor_index": int(neighbor_idx),
                "cosine_similarity": float(sim_row[neighbor_idx]),

                "query_crop_path": query_meta.get("crop_path", None),
                "query_original_crop_path": query_meta.get("original_crop_path", None),
                "query_source_type": query_meta.get("source_type", None),
                "query_quality_label": query_meta.get("quality_label", None),
                "query_dog_id": query_meta.get("dog_id", None),
                "query_report_id": query_meta.get("report_id", None),

                "neighbor_crop_path": neighbor_meta.get("crop_path", None),
                "neighbor_original_crop_path": neighbor_meta.get("original_crop_path", None),
                "neighbor_source_type": neighbor_meta.get("source_type", None),
                "neighbor_quality_label": neighbor_meta.get("quality_label", None),
                "neighbor_dog_id": neighbor_meta.get("dog_id", None),
                "neighbor_report_id": neighbor_meta.get("report_id", None),
            })

    return rows

In [ ]:
# 8. Ejecutar búsqueda Top-K por chunks

topk_records = []
start = time.time()

total_chunks = math.ceil(len(query_indices) / CHUNK_SIZE)

for chunk_start in tqdm(range(0, len(query_indices), CHUNK_SIZE), total=total_chunks):
    chunk_query_indices = query_indices[chunk_start:chunk_start + CHUNK_SIZE]
    query_vectors = embeddings_l2[chunk_query_indices]

    chunk_rows = topk_cosine_search(
        query_vectors=query_vectors,
        database_vectors=embeddings_l2,
        query_indices=chunk_query_indices,
        top_k=TOP_K
    )

    topk_records.extend(chunk_rows)

topk_neighbors_df = pd.DataFrame(topk_records)

elapsed = (time.time() - start) / 60

print("Búsqueda terminada.")
print("Tiempo:", round(elapsed, 2), "minutos")
print("Filas Top-K:", len(topk_neighbors_df))

display(topk_neighbors_df.head(20))

In [ ]:
# 9. Guardar resultados Top-K

TOPK_RESULTS_PATH = SEARCH_RESULTS_PATH / "step04_topk_neighbors.csv"
TOPK_RESULTS_TIMESTAMPED_PATH = SEARCH_RESULTS_PATH / f"step04_topk_neighbors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

topk_neighbors_df.to_csv(TOPK_RESULTS_PATH, index=False)
topk_neighbors_df.to_csv(TOPK_RESULTS_TIMESTAMPED_PATH, index=False)

# Copia en reports/tables
topk_neighbors_df.to_csv(TABLES_PATH / "step04_topk_neighbors.csv", index=False)

print("Top-K guardado en:", TOPK_RESULTS_PATH)
print("Top-K con timestamp guardado en:", TOPK_RESULTS_TIMESTAMPED_PATH)

## Métricas de búsqueda visual

En esta etapa todavía no estamos midiendo “mismo perro”, porque la base actual proviene principalmente del dataset base.  
Por eso medimos métricas generales de similitud visual:

- similitud promedio del Top-1,
- similitud promedio del Top-5,
- similitud promedio del Top-10,
- distribución de similitudes,
- casos con similitud alta,
- casos con similitud baja.

Cuando se agregue `identity_test`, se podrán medir métricas reales de re-identificación, como Top-1 Same Dog Accuracy y Top-5 Same Dog Accuracy.

In [ ]:
# 10. Métricas generales de similitud

top1_df = topk_neighbors_df[topk_neighbors_df["neighbor_rank"] == 1].copy()
top5_df = topk_neighbors_df[topk_neighbors_df["neighbor_rank"] <= 5].copy()
top10_df = topk_neighbors_df[topk_neighbors_df["neighbor_rank"] <= 10].copy()

avg_top1_similarity = float(top1_df["cosine_similarity"].mean()) if len(top1_df) > 0 else np.nan
avg_top5_similarity = float(top5_df["cosine_similarity"].mean()) if len(top5_df) > 0 else np.nan
avg_top10_similarity = float(top10_df["cosine_similarity"].mean()) if len(top10_df) > 0 else np.nan

median_top1_similarity = float(top1_df["cosine_similarity"].median()) if len(top1_df) > 0 else np.nan
min_top1_similarity = float(top1_df["cosine_similarity"].min()) if len(top1_df) > 0 else np.nan
max_top1_similarity = float(top1_df["cosine_similarity"].max()) if len(top1_df) > 0 else np.nan

high_similarity_cases = int((top1_df["cosine_similarity"] >= 0.90).sum()) if len(top1_df) > 0 else 0
possible_similarity_cases = int(((top1_df["cosine_similarity"] >= 0.80) & (top1_df["cosine_similarity"] < 0.90)).sum()) if len(top1_df) > 0 else 0
weak_similarity_cases = int(((top1_df["cosine_similarity"] >= 0.70) & (top1_df["cosine_similarity"] < 0.80)).sum()) if len(top1_df) > 0 else 0
low_similarity_cases = int((top1_df["cosine_similarity"] < 0.70).sum()) if len(top1_df) > 0 else 0

similarity_summary_df = pd.DataFrame([
    {"metric": "avg_top1_similarity", "value": avg_top1_similarity},
    {"metric": "avg_top5_similarity", "value": avg_top5_similarity},
    {"metric": "avg_top10_similarity", "value": avg_top10_similarity},
    {"metric": "median_top1_similarity", "value": median_top1_similarity},
    {"metric": "min_top1_similarity", "value": min_top1_similarity},
    {"metric": "max_top1_similarity", "value": max_top1_similarity},
    {"metric": "high_similarity_cases_top1_ge_0_90", "value": high_similarity_cases},
    {"metric": "possible_similarity_cases_top1_0_80_0_90", "value": possible_similarity_cases},
    {"metric": "weak_similarity_cases_top1_0_70_0_80", "value": weak_similarity_cases},
    {"metric": "low_similarity_cases_top1_lt_0_70", "value": low_similarity_cases},
])

SIMILARITY_SUMMARY_PATH = SEARCH_RESULTS_PATH / "step04_similarity_summary.csv"
similarity_summary_df.to_csv(SIMILARITY_SUMMARY_PATH, index=False)
similarity_summary_df.to_csv(TABLES_PATH / "step04_similarity_summary.csv", index=False)

display(similarity_summary_df)
print("Resumen de similitud guardado en:", SIMILARITY_SUMMARY_PATH)

In [ ]:
# 11. Indicadores finales del Paso 04

step04_indicators_df = pd.DataFrame([
    {"section": "input", "indicator": "num_embeddings", "value": embeddings_l2.shape[0]},
    {"section": "input", "indicator": "embedding_dimension", "value": embeddings_l2.shape[1]},
    {"section": "search", "indicator": "top_k", "value": TOP_K},
    {"section": "search", "indicator": "queries_processed", "value": len(query_indices)},
    {"section": "search", "indicator": "database_size", "value": embeddings_l2.shape[0]},
    {"section": "search", "indicator": "run_full_search", "value": RUN_FULL_SEARCH},
    {"section": "similarity", "indicator": "avg_top1_similarity", "value": avg_top1_similarity},
    {"section": "similarity", "indicator": "avg_top5_similarity", "value": avg_top5_similarity},
    {"section": "similarity", "indicator": "avg_top10_similarity", "value": avg_top10_similarity},
    {"section": "similarity", "indicator": "median_top1_similarity", "value": median_top1_similarity},
    {"section": "similarity", "indicator": "high_similarity_cases_top1_ge_0_90", "value": high_similarity_cases},
    {"section": "similarity", "indicator": "possible_similarity_cases_top1_0_80_0_90", "value": possible_similarity_cases},
    {"section": "similarity", "indicator": "weak_similarity_cases_top1_0_70_0_80", "value": weak_similarity_cases},
    {"section": "similarity", "indicator": "low_similarity_cases_top1_lt_0_70", "value": low_similarity_cases},
    {"section": "paths", "indicator": "topk_results_path", "value": str(TOPK_RESULTS_PATH)},
    {"section": "paths", "indicator": "similarity_summary_path", "value": str(SIMILARITY_SUMMARY_PATH)},
])

STEP04_INDICATORS_PATH = TABLES_PATH / "step04_vector_search_indicators.csv"
step04_indicators_df.to_csv(STEP04_INDICATORS_PATH, index=False)

display(step04_indicators_df)
print("Indicadores guardados en:", STEP04_INDICATORS_PATH)

In [ ]:
# 12. Histograma de similitud Top-1

if len(top1_df) > 0:
    plt.figure(figsize=(8, 4))
    plt.hist(top1_df["cosine_similarity"], bins=50)
    plt.title("Distribución de similitud coseno Top-1")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Frecuencia")
    plt.tight_layout()

    HIST_PATH = FIGURES_PATH / "step04_top1_similarity_distribution.png"
    plt.savefig(HIST_PATH, dpi=150)
    plt.show()

    print("Histograma guardado en:", HIST_PATH)
else:
    print("No hay datos Top-1 para graficar.")

## Visualización de vecinos

La siguiente sección muestra una imagen consulta y sus vecinos más similares.  
Para evitar problemas si las rutas locales de `/content` ya no existen, se usa preferentemente `original_crop_path`, que apunta a Google Drive.

In [ ]:
# 13. Funciones de visualización de vecinos

def resolve_image_path(path_a, path_b=None):
    """Devuelve una ruta existente priorizando path_a y luego path_b."""
    for p in [path_a, path_b]:
        if p is None or pd.isna(p):
            continue
        p = Path(str(p))
        if p.exists():
            return str(p)
    return None


def show_query_neighbors(query_index, topk_df, metadata_df, top_k=5, save_path=None):
    query_meta = metadata_df.iloc[int(query_index)]

    query_img_path = resolve_image_path(
        query_meta.get("original_crop_path", None),
        query_meta.get("crop_path", None)
    )

    neighbors = (
        topk_df[
            (topk_df["query_index"] == int(query_index)) &
            (topk_df["neighbor_rank"] <= top_k)
        ]
        .sort_values("neighbor_rank")
    )

    image_paths = [query_img_path]
    titles = [f"Query\\nidx={query_index}"]

    for _, row in neighbors.iterrows():
        neighbor_path = resolve_image_path(
            row.get("neighbor_original_crop_path", None),
            row.get("neighbor_crop_path", None)
        )
        image_paths.append(neighbor_path)
        titles.append(
            f"Rank {int(row['neighbor_rank'])}\\n"
            f"cos={row['cosine_similarity']:.3f}"
        )

    cols = len(image_paths)
    plt.figure(figsize=(3.2 * cols, 3.8))

    for i, img_path in enumerate(image_paths):
        plt.subplot(1, cols, i + 1)

        if img_path is None:
            plt.text(0.5, 0.5, "Imagen no encontrada", ha="center", va="center")
            plt.axis("off")
            continue

        try:
            img = Image.open(img_path).convert("RGB")
            plt.imshow(img)
            plt.title(titles[i], fontsize=9)
            plt.axis("off")
        except Exception as e:
            plt.text(0.5, 0.5, f"Error\\n{e}", ha="center", va="center")
            plt.axis("off")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150)

    plt.show()

In [ ]:
# 14. Mostrar ejemplos de búsqueda visual

NUM_EXAMPLES = 5

if len(top1_df) > 0:
    # Mezclar ejemplos: algunos de alta similitud y algunos aleatorios.
    candidate_queries = top1_df.sort_values("cosine_similarity", ascending=False)["query_index"].head(50).tolist()
    random_queries = top1_df.sample(min(50, len(top1_df)), random_state=SEED)["query_index"].tolist()

    example_queries = list(dict.fromkeys(candidate_queries + random_queries))[:NUM_EXAMPLES]

    for qidx in example_queries:
        save_path = FIGURES_PATH / f"step04_query_{qidx}_neighbors.png"
        show_query_neighbors(qidx, topk_neighbors_df, metadata_df, top_k=VISUAL_TOP_K, save_path=save_path)
        print("Ejemplo guardado en:", save_path)
else:
    print("No hay resultados Top-K para visualizar.")

In [ ]:
# 15. Casos de baja similitud para revisión

if len(top1_df) > 0:
    low_similarity_review_df = top1_df.sort_values("cosine_similarity", ascending=True).head(50).copy()

    LOW_SIMILARITY_REVIEW_PATH = SEARCH_RESULTS_PATH / "step04_low_similarity_review_cases.csv"
    low_similarity_review_df.to_csv(LOW_SIMILARITY_REVIEW_PATH, index=False)

    display(low_similarity_review_df.head(10))
    print("Casos de baja similitud guardados en:", LOW_SIMILARITY_REVIEW_PATH)
else:
    print("No hay resultados Top-1.")

# Análisis de métricas del Paso 04

El Paso 04 evalúa si los embeddings generados en el Paso 03 permiten recuperar imágenes visualmente similares.

Las métricas de similitud Top-1, Top-5 y Top-10 permiten entender qué tan cerca está cada imagen de sus vecinos más parecidos. Una similitud Top-1 alta indica que el sistema encontró una imagen visualmente muy cercana al query. Una similitud baja puede indicar un caso difícil, una imagen atípica, un crop de baja calidad o un perro con características menos representadas en la base.

En esta etapa todavía no se debe interpretar la búsqueda como identificación definitiva del mismo perro. Para eso se necesita un conjunto `identity_test`, donde varias imágenes pertenezcan al mismo perro. Sin embargo, este paso valida que la base vectorial funciona y que el sistema puede recuperar vecinos visualmente similares.

# Conclusión del Paso 04

Este notebook construye la primera versión de búsqueda visual por embeddings para `proyecto_integrador_v2`.

A partir de los embeddings normalizados del Paso 03, se generó una búsqueda Top-K mediante similitud coseno. Los resultados permiten encontrar los perros visualmente más parecidos a una imagen consulta.

Este paso convierte el proyecto en una base de búsqueda visual. La raza deja de ser el elemento central; ahora el componente principal es la comparación entre vectores visuales.